In [2]:
import numpy as np
from scipy.optimize import linprog
from queue import Queue

# -------------------------------
# Function to solve LP relaxation
def solve_lp(c, A, b, bounds):
    return linprog(c, A_ub=A, b_ub=b, bounds=bounds, method='highs')

# -------------------------------
# Branch and Bound Method
def branch_and_bound(c, A, b, bounds):
    Q = Queue()
    Q.put((c, A, b, bounds))

    best_solution = None
    best_value = float('-inf')

    while not Q.empty():
        current_problem = Q.get()
        res = solve_lp(*current_problem)

        if res.success:
            value = -res.fun  # convert back to maximization

            if value > best_value:
                solution = res.x

                # Check if integer solution
                if all(np.isclose(solution, np.round(solution))):
                    best_value = value
                    best_solution = solution
                else:
                    # Branching
                    for i in range(len(solution)):
                        if not np.isclose(solution[i], np.round(solution[i])):
                            
                            lower_bounds = current_problem[3].copy()
                            upper_bounds = current_problem[3].copy()

                            lower_bounds[i] = (lower_bounds[i][0], np.floor(solution[i]))
                            upper_bounds[i] = (np.ceil(solution[i]), upper_bounds[i][1])

                            Q.put((c, A, b, lower_bounds))
                            Q.put((c, A, b, upper_bounds))
                            break

    return best_solution, best_value

# -------------------------------
# Example Problem
c = [-6, -5]  # Maximize → convert to minimize

A = [
    [3, 2],
    [1, 2]
]

b = [12, 8]

bounds = [(0, None), (0, None)]

# -------------------------------
# Solve
solution, value = branch_and_bound(c, A, b, bounds)

# -------------------------------
# Output
print("Optimal solution (x, y):", solution)
print("Optimal value (Z):", value)

Optimal solution (x, y): [2. 3.]
Optimal value (Z): 27.0
